# **Загрузка набора данных ["Avazu Click-Through Rate Prediction"](https://www.kaggle.com/competitions/avazu-ctr-prediction/overview) в ClickHouse**

**Проект:** Анализ и визуализация данных с использованием Yandex DataLens: исследование по прогнозированию CTR

**Автор:** Грицан М.А., студент группы БПМИ-247, 2 курса

**Дата:** 06-04-2026  

**Цель:** Отправить Kaggle набор данных в ClickHouse для последующего подключения базы данных к Yandex Datalens

#### Для начала установим подключение к ClickHouse:

In [25]:
!pip install clickhouse_connect -q

In [ ]:
import clickhouse_connect

client = clickhouse_connect.get_client(
    host='ВАШ_HOST',
    port=8443,
    username='ВАШ_USERNAME',
    password='ВАШ_PASSWORD',
    secure=True
)

print("✅ Подкючение установленно!")

✅ Подкючение установленно!


#### (Пере)создадим таблицу, в которую будем загружать датасет дальше:

In [27]:
client.command('DROP TABLE IF EXISTS avazu_ctr_prediction_data')

client.command('''
CREATE TABLE avazu_ctr_prediction_data
(
    id String,
    click UInt8,
    hour UInt64,
    C1 Int32,
    banner_pos Int32,
    site_id String,
    site_domain String,
    site_category String,
    app_id String,
    app_domain String,
    app_category String,
    device_id String,
    device_ip String,
    device_model String,
    device_type Int32,
    device_conn_type Int32,
    C14 Int32,
    C15 Int32,
    C16 Int32,
    C17 Int32,
    C18 Int32,
    C19 Int32,
    C20 Int32,
    C21 Int32
)
ENGINE = MergeTree
ORDER BY (hour, id);
''')

print("✅ Таблица пересоздана!")

✅ Таблица пересоздана!


#### А теперь начнем обработку датасета и отправку батчей с нужными строками:


In [28]:
import pandas as pd

chunksize = 1_000_000
file_path = 'avazu-ctr-prediction/train.gz'
data_fraction = 0.2

string_columns = [
    'id', 'site_id', 'site_domain', 'site_category',
    'app_id', 'app_domain', 'app_category',
    'device_id', 'device_ip', 'device_model'
]

dtype_dict = {col: str for col in string_columns}

print("⏳ Начинаем загрузку набора данных в Clickhouse...")
for i, chunk in enumerate(pd.read_csv(file_path, compression='gzip', chunksize=chunksize, dtype=dtype_dict)):
    sampled_chunk = chunk.sample(frac=data_fraction, random_state=67)

    for col in string_columns:
        sampled_chunk[col] = sampled_chunk[col].fillna('')

    client.insert_df('avazu_ctr_prediction_data', sampled_chunk)
    print(f" Обработано {i+1} млн. строк из датасета")

print("\n✅ Набор данных успешно загружен!")

⏳ Начинаем загрузку набора данных в Clickhouse...
 Обработано 1 млн. строк из датасета
 Обработано 2 млн. строк из датасета
 Обработано 3 млн. строк из датасета
 Обработано 4 млн. строк из датасета
 Обработано 5 млн. строк из датасета
 Обработано 6 млн. строк из датасета
 Обработано 7 млн. строк из датасета
 Обработано 8 млн. строк из датасета
 Обработано 9 млн. строк из датасета
 Обработано 10 млн. строк из датасета
 Обработано 11 млн. строк из датасета
 Обработано 12 млн. строк из датасета
 Обработано 13 млн. строк из датасета
 Обработано 14 млн. строк из датасета
 Обработано 15 млн. строк из датасета
 Обработано 16 млн. строк из датасета
 Обработано 17 млн. строк из датасета
 Обработано 18 млн. строк из датасета
 Обработано 19 млн. строк из датасета
 Обработано 20 млн. строк из датасета
 Обработано 21 млн. строк из датасета
 Обработано 22 млн. строк из датасета
 Обработано 23 млн. строк из датасета
 Обработано 24 млн. строк из датасета
 Обработано 25 млн. строк из датасета
 Обработа